In [ ]:
# Import the required packages.
import sys
import json
import numbers
from pathlib import Path
import networkx as nx
import pandas as pd

print(sys.executable)
print("networkx:", nx.__version__)
print("pandas:", pd.__version__)

D:\anaconda\python.exe
networkx: 3.4.2
pandas: 2.2.3


In [2]:
# Load the graph and metadata.
input_dir = Path("raw data")
output_dir = Path(".")
G_raw = nx.read_gml(input_dir / "political_network.gml")

with open(input_dir / "MP.json", encoding="utf-8-sig") as f:
    mp_data = json.load(f)
with open(input_dir / "Party.json", encoding="utf-8-sig") as f:
    party_data = json.load(f)

print("Graph:", G_raw.number_of_nodes(), "nodes,", G_raw.number_of_edges(), "edges")
print("MP records:", len(mp_data))
print("Party records:", len(party_data))

Graph: 365 nodes, 9617 edges
MP records: 365
Party records: 38


In [3]:
# Check the raw graph statistics.
def check_expected(label, actual, expected):
    if actual != expected:
        print(f"WARNING: {label} is {actual}; expected {expected}")

raw_nodes = G_raw.number_of_nodes()
raw_edges = G_raw.number_of_edges()
raw_self_loops = nx.number_of_selfloops(G_raw)
components_raw = list(nx.weakly_connected_components(G_raw) if G_raw.is_directed() else nx.connected_components(G_raw))
component_sizes_raw = sorted((len(c) for c in components_raw), reverse=True)
missing_ids = sorted(set(G_raw.nodes) - set(mp_data))
missing_party_before = sum(not mp_data[n].get("PartyName") or mp_data[n].get("PartyID") in (0, "0", None, "") for n in G_raw.nodes if n in mp_data)

weights = []
invalid_weights = []
for source, target, edge_data in G_raw.edges(data=True):
    weight = edge_data.get("weight")
    if not isinstance(weight, numbers.Number) or isinstance(weight, bool) or weight <= 0:
        invalid_weights.append((source, target, weight))
    else:
        weights.append(weight)

min_weight = min(weights) if weights else None
max_weight = max(weights) if weights else None

checks = [("nodes", raw_nodes, 365), ("edges", raw_edges, 9617), ("directed", G_raw.is_directed(), False), ("multigraph", G_raw.is_multigraph(), False), ("components", len(components_raw), 3), ("component sizes", component_sizes_raw, [355, 9, 1]), ("self-loops", raw_self_loops, 5), ("missing parties", missing_party_before, 16), ("minimum weight", min_weight, 1), ("maximum weight", max_weight, 369)]
for label, actual, expected in checks:
    check_expected(label, actual, expected)

if missing_ids:
    print("WARNING: unmatched node IDs:", missing_ids)
if invalid_weights:
    print("WARNING: invalid edge weights:", invalid_weights[:10])

print("Component sizes:", component_sizes_raw)
print("Unmatched node IDs:", len(missing_ids))
print("Weight range:", min_weight, "to", max_weight)

Component sizes: [355, 9, 1]
Unmatched node IDs: 0
Weight range: 1 to 369


In [4]:
# Add readable MP metadata.
if missing_ids:
    raise ValueError("Some network nodes cannot be matched to MP.json")
if invalid_weights:
    raise ValueError("All edge weights must be numeric and positive")

G = G_raw.copy()
for node_id in G.nodes:
    record = mp_data[node_id]
    G.nodes[node_id]["name"] = record.get("MP", "")
    G.nodes[node_id]["party"] = record.get("PartyName") or "Unknown"
    G.nodes[node_id]["party_id"] = record.get("PartyID", 0)

missing_party_after = sum(G.nodes[n]["party"] == "Unknown" for n in G.nodes)
print("Party labels mapped to Unknown:", missing_party_after)

Party labels mapped to Unknown: 16


In [5]:
# Check duplicate edges and remove self-loops.
if G.is_multigraph():
    combined = nx.Graph()
    combined.add_nodes_from(G.nodes(data=True))
    duplicate_edges = 0
    for source, target, edge_data in G.edges(data=True):
        if combined.has_edge(source, target):
            combined[source][target]["weight"] += edge_data["weight"]
            duplicate_edges += 1
        else:
            combined.add_edge(source, target, **edge_data)
    G = combined
    print(f"Combined {duplicate_edges} parallel edge(s).")
else:
    pairs = [frozenset((a, b)) for a, b in G.edges() if a != b]
    duplicate_edges = len(pairs) - len(set(pairs))
    print("Duplicate/parallel edges found:", duplicate_edges)

self_loops = list(nx.selfloop_edges(G))
G.remove_edges_from(self_loops)
print("Self-loops removed:", len(self_loops))
print("Self-loops remaining:", nx.number_of_selfloops(G))

Duplicate/parallel edges found: 0
Self-loops removed: 5
Self-loops remaining: 0


In [6]:
# Create and save the cleaned outputs.
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]
G_lcc = G.subgraph(components[0]).copy()

for item in [("clean nodes", G.number_of_nodes(), 365), ("clean edges", G.number_of_edges(), 9612), ("component sizes", component_sizes, [355, 9, 1]), ("LCC nodes", G_lcc.number_of_nodes(), 355), ("LCC edges", G_lcc.number_of_edges(), 9588)]:
    check_expected(*item)

nx.write_gml(G, output_dir / "political_network_clean_full.gml")
nx.write_gml(G_lcc, output_dir / "political_network_clean_lcc.gml")

nodes_df = pd.DataFrame([{"node_id": n, "name": d["name"], "party": d["party"], "party_id": d["party_id"]} for n, d in G.nodes(data=True)]).sort_values("node_id")
edges_df = pd.DataFrame([{"source": min(a, b), "target": max(a, b), "weight": d["weight"]} for a, b, d in G.edges(data=True)]).sort_values(["source", "target"])
nodes_df.to_csv(output_dir / "nodes_clean.csv", index=False, encoding="utf-8")
edges_df.to_csv(output_dir / "edges_clean.csv", index=False, encoding="utf-8")
print("Saved two GML files and two full-graph CSV files.")

Saved two GML files and two full-graph CSV files.


In [7]:
# Print the final summary.
print(f"""Raw graph
---------
Nodes: {raw_nodes}
Edges: {raw_edges}
Connected components: {len(components_raw)}
Component sizes: {component_sizes_raw}
Self-loops: {raw_self_loops}
Missing party labels: {missing_party_before}
Unmatched node IDs: {len(missing_ids)}
Edge weight range: {min_weight} to {max_weight}

Clean full graph
----------------
Nodes: {G.number_of_nodes()}
Edges: {G.number_of_edges()}
Self-loops: {nx.number_of_selfloops(G)}
Party labels mapped to Unknown: {missing_party_after}

Largest connected component
---------------------------
Nodes: {G_lcc.number_of_nodes()}
Edges: {G_lcc.number_of_edges()}""")

Raw graph
---------
Nodes: 365
Edges: 9617
Connected components: 3
Component sizes: [355, 9, 1]
Self-loops: 5
Missing party labels: 16
Unmatched node IDs: 0
Edge weight range: 1 to 369

Clean full graph
----------------
Nodes: 365
Edges: 9612
Self-loops: 0
Party labels mapped to Unknown: 16

Largest connected component
---------------------------
Nodes: 355
Edges: 9588
